# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
import math
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

import nltk
for package in ('gutenberg', 'stopwords'):
    try:
        nltk.data.find(f'corpora/{package}')
    except LookupError:
        nltk.download(package, quiet=True)

from nltk.corpus import gutenberg

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 160)

STOPWORDS = set('''a about above after again against ain all am an and any are aren aren't as at be because been before being below between both but by can couldn couldn't d did didn didn't do does doesn doesn't doing don don't down during each few for from further had hadn hadn't has hasn hasn't have haven haven't having he he'd he'll he's her here hers herself him himself his how i i'd i'll i'm i've if in into is isn isn't it it'd it'll it's its itself just ll m ma me mightn mightn't more most must mustn't my myself needn needn't no nor not now o of off on once only or other our ours ourselves out over own re s same shan shan't she she'd she'll she's should should've shouldn shouldn't so some such t than that that'll the their theirs them themselves then there these they they'd they'll they're they've this those through to too under until up ve very was wasn wasn't we we'd we'll we're we've were weren weren't what when where which while who whom why will with won won't wouldn wouldn't y you you'd you'll you're you've your yours yourself yourselves'''.split())
TOKEN_RE = re.compile(r'[a-z]+')


def normalize_text(text):
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch)).lower()
    start = text.find('*** start of')
    end = text.find('*** end of')
    if start != -1:
        text = text[start:]
    if end != -1:
        text = text[:end]
    return text


def tokenize(text):
    return [
        token
        for token in TOKEN_RE.findall(normalize_text(text))
        if len(token) >= 3 and token not in STOPWORDS
    ]


def extract_title(raw_text, fallback):
    first_line = raw_text.splitlines()[0].strip()
    if first_line.startswith('[') and first_line.endswith(']'):
        content = first_line[1:-1]
        return re.split(r',?\s+by\s+', content)[0].strip()
    return fallback


doc_ids = list(gutenberg.fileids())
raw_docs = {doc_id: gutenberg.raw(doc_id) for doc_id in doc_ids}
tokenized_docs = {doc_id: tokenize(raw_docs[doc_id]) for doc_id in doc_ids}
vocabulary = sorted({term for tokens in tokenized_docs.values() for term in tokens})

doc_table = pd.DataFrame(
    {
        'Identificador': doc_ids,
        'Título': [extract_title(raw_docs[doc_id], doc_id) for doc_id in doc_ids],
        'Términos únicos': [len(set(tokenized_docs[doc_id])) for doc_id in doc_ids],
    }
)
display(doc_table)

doc_lengths = np.array([len(tokenized_docs[doc_id]) for doc_id in doc_ids])
print(f'Documentos: {len(doc_ids)}')
print(f'Tamaño del vocabulario: {len(vocabulary):,}')
print(f'Longitud mínima: {doc_lengths.min():,} términos')
print(f'Longitud media: {doc_lengths.mean():,.2f} términos')
print(f'Longitud máxima: {doc_lengths.max():,} términos')

,Identificador,Título,Términos únicos
0,austen-emma.txt,Emma,6931
1,austen-persuasion.txt,Persuasion,5586
2,austen-sense.txt,Sense and Sensibility,6128
3,bible-kjv.txt,The King James Bible,12424
4,blake-poems.txt,Poems,1386
5,bryant-stories.txt,Stories to Tell to Children,3675
6,burgess-busterbrown.txt,The Adventures of Buster Bear,1356
7,carroll-alice.txt,Alice's Adventures in Wonderland,2402
8,chesterton-ball.txt,The Ball and The Cross,7686
9,chesterton-brown.txt,The Wisdom of Father Brown,7558


Documentos: 18
Tamaño del vocabulario: 41,126
Longitud mínima: 3,742 términos
Longitud media: 55,850.56 términos
Longitud máxima: 367,512 términos


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [2]:
def compute_tf(tokens):
    return Counter(tokens)


def compute_df(tokenized_docs, vocabulary):
    df = {term: 0 for term in vocabulary}
    for tokens in tokenized_docs.values():
        for term in set(tokens):
            df[term] += 1
    return df


def compute_idf(df, n_documents, add_one=True):
    if add_one:
        return {
            term: math.log(n_documents / freq) + 1
            for term, freq in df.items()
        }
    return {
        term: math.log(n_documents / freq)
        for term, freq in df.items()
    }


def tf_weight(count, scheme, max_count):
    if count == 0:
        return 0.0
    if scheme == 'raw':
        return float(count)
    if scheme == 'log':
        return 1.0 + math.log(count)
    if scheme == 'augmented':
        return 0.5 + 0.5 * count / max_count
    raise ValueError(f'Esquema TF desconocido: {scheme}')


TF_SCHEMES = ('raw', 'log', 'augmented')
IDF_SCHEMES = ('classic', 'plus_one')
SIX_TFIDF_WEIGHTS = [
    (tf_scheme, idf_scheme)
    for tf_scheme in TF_SCHEMES
    for idf_scheme in IDF_SCHEMES
]


def compute_tfidf(tokenized_docs, vocabulary, tf_scheme='raw', idf_scheme='plus_one'):
    doc_ids = list(tokenized_docs)
    term_index = {term: i for i, term in enumerate(vocabulary)}
    n_documents = len(doc_ids)
    df = compute_df(tokenized_docs, vocabulary)
    idf = compute_idf(df, n_documents, add_one=(idf_scheme == 'plus_one'))
    matrix = np.zeros((n_documents, len(vocabulary)), dtype=float)
    for row, doc_id in enumerate(doc_ids):
        counts = compute_tf(tokenized_docs[doc_id])
        max_count = max(counts.values()) if counts else 0
        for term, count in counts.items():
            weight = tf_weight(count, tf_scheme, max_count) * idf[term]
            matrix[row, term_index[term]] = weight
    return matrix, df, idf


print('Seis pesos TF-IDF disponibles:')
for tf_scheme in TF_SCHEMES:
    for idf_scheme in IDF_SCHEMES:
        print(f'  TF={tf_scheme:9s} IDF={idf_scheme}')

sample_doc = doc_ids[0]
sample_counts = compute_tf(tokenized_docs[sample_doc])
sample_term = sample_counts.most_common(1)[0][0]
sample_max_count = max(sample_counts.values())
df_all = compute_df(tokenized_docs, vocabulary)
idf_classic = compute_idf(df_all, len(doc_ids), add_one=False)
idf_plus_one = compute_idf(df_all, len(doc_ids), add_one=True)

rows = []
for tf_scheme in TF_SCHEMES:
    for idf_scheme in IDF_SCHEMES:
        idf_values = idf_plus_one if idf_scheme == 'plus_one' else idf_classic
        rows.append(
            {
                'Documento': sample_doc,
                'Término': sample_term,
                'TF': tf_scheme,
                'IDF': idf_scheme,
                'Peso TF-IDF': tf_weight(sample_counts[sample_term], tf_scheme, sample_max_count)
                * idf_values[sample_term],
            }
        )
display(pd.DataFrame(rows))

Seis pesos TF-IDF disponibles:
  TF=raw       IDF=classic
  TF=raw       IDF=plus_one
  TF=log       IDF=classic
  TF=log       IDF=plus_one
  TF=augmented IDF=classic
  TF=augmented IDF=plus_one


,Documento,Término,TF,IDF,Peso TF-IDF
0,austen-emma.txt,emma,raw,classic,1900.599259
1,austen-emma.txt,emma,raw,plus_one,2765.599259
2,austen-emma.txt,emma,log,classic,17.056460
3,austen-emma.txt,emma,log,plus_one,24.819190
4,austen-emma.txt,emma,augmented,classic,2.197225
5,austen-emma.txt,emma,augmented,plus_one,3.197225


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [3]:
vocabulary = sorted({term for tokens in tokenized_docs.values() for term in tokens})
term_index = {term: i for i, term in enumerate(vocabulary)}

tf_counts = {doc_id: compute_tf(tokenized_docs[doc_id]) for doc_id in doc_ids}

df = compute_df(tokenized_docs, vocabulary)

idf = compute_idf(df, len(doc_ids), add_one=True)

tfidf_matrix = np.zeros((len(doc_ids), len(vocabulary)), dtype=float)
for row, doc_id in enumerate(doc_ids):
    for term, count in tf_counts[doc_id].items():
        tfidf_matrix[row, term_index[term]] = count * idf[term]

print(f'Documentos (filas): {tfidf_matrix.shape[0]}')
print(f'Términos (columnas): {tfidf_matrix.shape[1]:,}')
print(f'Valores no nulos: {np.count_nonzero(tfidf_matrix):,}')
print(f'Máximo peso TF-IDF: {tfidf_matrix.max():,.4f}')

sample_terms = ['love', 'whale', 'moby', 'sea']
sample_index = [term_index[term] for term in sample_terms]
display(
    pd.DataFrame(
        tfidf_matrix[:, sample_index],
        index=doc_ids,
        columns=sample_terms,
    ).loc[lambda frame: frame.sum(axis=1) > 0]
)

Documentos (filas): 18
Términos (columnas): 41,126
Valores no nulos: 117,854
Máximo peso TF-IDF: 18,881.2148


,love,whale,moby,sea
austen-emma.txt,138.331622,0.000000,0.000000,15.648962
austen-persuasion.txt,49.657505,0.000000,0.000000,23.473444
austen-sense.txt,91.038760,0.000000,0.000000,0.000000
bible-kjv.txt,367.702004,6.295837,0.000000,447.113214
blake-poems.txt,34.287325,0.000000,0.000000,1.117783
bryant-stories.txt,15.370180,33.577797,0.000000,22.355661
burgess-busterbrown.txt,1.182322,0.000000,0.000000,0.000000
carroll-alice.txt,3.546965,0.000000,0.000000,15.648962
chesterton-ball.txt,29.558039,4.197225,0.000000,63.713633
chesterton-brown.txt,10.640894,0.000000,0.000000,69.302548


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.